# RCSB PDB Custom Report Curation & Metadata Cleaning Pipeline

This notebook demonstrates how to use the **** suite to clean, forward-fill, merge, and summarize sequence and structure custom reports exported from RCSB PDB for Transcription Factor (TF) assemblies.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

# Add utils directory to Python path
project_root = Path("..").resolve()
utils_dir = project_root / "utils"
if str(utils_dir) not in sys.path:
    sys.path.insert(0, str(utils_dir))

from utils import (
    read_rcsb_custom_report,
    combine_asym_ids,
    filter_complete_complexes,
    abbreviate_organism,
    calculate_bsa_from_int,
    build_complex_table,
    clean_and_merge_custom_reports
)

print("Successfully imported utils module functions!")

## Step 1: Read Custom Reports

RCSB PDB custom report CSV files often have multi-line headers or multi-entity entry structures.  automatically detects 1-header or 2-header RCSB CSV structures and parses them into pandas DataFrames.

In [ ]:
revision_dir = Path("/home/labuser/Projects/PhD_projects/swarnava_TF_work/Interface/Revision")
struct_csv = revision_dir / "TFNRD_EM_structure_custom_report.csv"
seq_csv = revision_dir / "TFNRD_EM_sequence.csv"

struct_df = read_rcsb_custom_report(struct_csv)
seq_df = read_rcsb_custom_report(seq_csv)

print(f"Raw Structure Custom Report shape: {struct_df.shape}")
print(f"Raw Sequence Custom Report shape: {seq_df.shape}")
struct_df.head(5)

## Step 2: Combine Chain IDs for Identical Macromolecule Entities

When multiple chains share the same sequence and entity within an entry,  merges their chain IDs (e.g. ,  -> ).

In [ ]:
# Forward-fill Entry ID and entry-level metadata
struct_df['Entry ID'] = struct_df['Entry ID'].ffill()

for col in ['Refinement Resolution (Å)', 'Experimental Method', 'Source Organism', 'Oligomeric State']:
    if col in struct_df.columns:
        struct_df[col] = struct_df[col].ffill()

# Combine chains for protein, DNA, and RNA
combined_df = struct_df.copy()
for cond in [{'Entity Macromolecule Type': 'polypeptide(L)'},
             {'Entity Macromolecule Type': 'polydeoxyribonucleotide'},
             {'Entity Macromolecule Type': 'polyribonucleotide'}]:
    combined_df = combine_asym_ids(combined_df, cond)

print(f"Combined Chains dataset shape: {combined_df.shape}")
combined_df[['Entry ID', 'Auth Asym ID', 'Entity Macromolecule Type', 'Macromolecule Name']].head(10)

## Step 3: Filter Complete Complexes

Retain entries that contain both a qualifying protein (sequence length >= 30) AND nucleic acid (sequence length >= 5).

In [ ]:
filtered_df = filter_complete_complexes(combined_df, min_protein_len=30, min_na_len=5)
print(f"Filtered complete complexes shape: {filtered_df.shape}")
print(f"Unique Entry IDs: {filtered_df['Entry ID'].nunique()}")

## Step 4: Build Complex Summary Table with Entity Counts & BSA Metrics

Construct a 1-row-per-Entry-ID summary table including:
- Entity counts: ****, ****, ****
- Assembly metadata: ****, , 
- Interface BSA metrics: ****, ****, ****, ****

In [ ]:
prince_dir = revision_dir / "prince_results"
complex_tbl = build_complex_table(filtered_df, prince_results_dir=prince_dir)
print(f"Complex Summary Table shape: {complex_tbl.shape}")
complex_tbl.head(10)

## Step 5: End-to-End Master Pipeline Execution

Execute the master  function to perform all steps in a single automated call.

In [ ]:
merged_df, complex_df = clean_and_merge_custom_reports(struct_csv, seq_csv, prince_results_dir=prince_dir)

print(f"Master Cleaned Merged Metadata shape: {merged_df.shape}")
print(f"Master Complex Summary Table shape: {complex_df.shape}")

# Export cleaned outputs
out_csv = revision_dir / "TFNRDv1.0_EM_complex_table.csv"
out_xlsx = revision_dir / "TFNRDv1.0_EM_complex_table.xlsx"

complex_df.to_csv(out_csv, index=False)
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    complex_df.to_excel(writer, index=False, sheet_name='Complex_Table')

print(f"Successfully exported complex summary table to {out_csv} and {out_xlsx}")